# Colab Pro GRPO Demo — Bioresearch (Qwen2.5-7B, Auto-detect A100/L4, Hackathon Edition)

<a href="https://colab.research.google.com/github/<GITHUB_OWNER>/bioresearch/blob/main/notebooks/train_grpo_colab_pro.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Hackathon quality run.** This notebook squeezes the maximum number of GRPO epochs out of a Colab Pro session on Qwen2.5-7B — **~8.6 epochs on A100 40 GB** or **~6.4 epochs on L4 22.5 GB** — meaningfully more than the L40s notebook's 5.4 and the original A100 notebook's 7.1. Every modern fix from the T4 / L40s patch sets is bundled in (modern `huggingface_hub`/`trl` pins, the Qwen `generation_config.max_length` patch, no vLLM, cosine LR schedule + 5% warmup, `save_steps=30` for resilience).

Engineered around the hackathon's two weakest-typically axes:

- **Observable evidence of training progress (20%):**
  - **Live Trackio dashboard** that the judges can watch during the demo (auto-syncs to a HF Space if `HF_TOKEN` + `HF_SPACE_ID` are set, so it persists after the runtime dies).
  - **Baseline-vs-trained eval block** that runs the *exact same* held-out `task_id`s before training and after training, then renders a side-by-side table + sample rollout transcripts.
- **Reward coherent + meaningful inference improvement (10%):**
  - **Reward-distribution diagnostic** sampled before training so we can see the reward signal has variance (not stuck at 0 or 1) — without this the GRPO advantage collapses.
  - **`LabShapingCallback`** decomposes lab-task reward into *process* (per-step shaping) and *terminal* (final outcome) so the curves show whether training is learning to *think* better or just to *guess* better.

All heavy logic lives in [`training_core.py`](../training_core.py), [`training_a100.py`](../training_a100.py), and the new [`training_colab_pro.py`](../training_colab_pro.py) (auto-detect + per-card presets). This notebook is a thin driver.

---

## Why Qwen 7B at max epochs (and not 14B)

We hard-pin **Qwen2.5-7B-Instruct (4-bit)** and spend every Colab Pro minute on **more epochs**, not a bigger model:

- **GRPO teaches *behavior*, not knowledge.** The reward function gates on valid JSON, correct phase transitions, and right tool calls — 7B already has the biology priors it needs. The L40s and A100 reward curves confirm: format/phase/tool-call axes saturate well below 14B's ceiling.
- **14B doubles wall-clock for a ~3-5 pp lift on knowledge-bound tasks only** (`protein_function`, `kegg_pathway_reasoning`). On a Colab Pro session that's the difference between fitting 6 epochs vs 9 epochs of the *behavior-shaping* training that GRPO actually does.
- **Epochs are the real lever for this dataset.** With 112 prompts (14 tasks × `n_per_task=8`) and 4 prompts/step, every 28 steps = 1 epoch. The L40s preset stops at 5.4 epochs (150 steps); the original A100 notebook reaches 7.1 (200 steps). On Colab Pro we go to **8.6 on A100** (240 steps) or **6.4 on L4** (180 steps) — both sit just inside the no-overfit regime for GRPO with KL `beta=0.04`.
- **`num_generations` is bumped per card.** GRPO's group-relative advantage variance scales as `1/N`, so the A100 preset uses N=12 (vs the original A100's N=8) and L4 uses N=8. Tighter advantage estimates → smoother reward curves.

---

## Auto-GPU detection

Colab Pro hands out either an A100 (Pro+ priority) or an L4 (Pro tier or A100 spillover). A notebook that crashes when it gets the "wrong" GPU is a hackathon-day disaster, so [`training_colab_pro.detect_gpu`](../training_colab_pro.py) reads `torch.cuda.get_device_name` at runtime and picks the right preset:

```mermaid
flowchart LR
  start([Cell 8 boot]) --> detect{detect_gpu}
  detect -->|"A100 (sm_80, 40 GB)"| pa[preset_a100<br/>num_gen=12 · max_steps=240<br/>max_completion=768 · ~8.6 epochs]
  detect -->|"L4 (sm_89, 22.5 GB)"| pl[preset_l4<br/>num_gen=8 · max_steps=180<br/>max_completion=640 · ~6.4 epochs]
  detect -->|"L40S (sm_89, 48 GB)"| ps[preset_l40s<br/>num_gen=10 · max_steps=200]
  detect -->|"H100 / unknown"| pf[fallback: preset_l4<br/>conservative, never OOMs]
  pa --> trainer
  pl --> trainer
  ps --> trainer
  pf --> trainer
  trainer([GRPOConfig + GRPOTrainer])
```

---

## Wall-clock targets (full 14-task run)

| GPU | Train | Eval (baseline + trained) | **Total** | Epochs |
|-----|-------|---------------------------|-----------|--------|
| **A100 40 GB** | ~3 hr (240 steps × ~45-55 s) | ~15 min | **~3 h 15 m** | 8.6 |
| **L4 22.5 GB** | ~2 h 30 m (180 steps × ~50-55 s) | ~15 min | **~2 h 45 m** | 6.4 |

Both fit comfortably inside Colab Pro's session window. `save_steps=30` means a checkpoint lands every ~12 min on A100 / ~17 min on L4, so a mid-run disconnect costs at most one checkpoint.

---

## Hackathon checklist (artifacts produced)

- `baseline_rollouts.json` + `trained_rollouts.json` — paired before/after rollouts on identical `task_id`s
- `reward_diagnostic.png` — proof the reward signal has variance for GRPO to optimize
- `before_after_bar.png` — per-task mean reward delta with the headline numbers annotated
- `sample_transcripts.md` — top-2 winning side-by-side rollouts per task (judges read this)
- `training_curves.png` — per-task reward EMA over training steps
- `grpo_bioresearch_colab_pro_lora/` — final LoRA adapter (~150 MB, ready for Hub push)
- Live Trackio dashboard while training (auto-syncs to HF Space if configured)

## 1 · Install dependencies

In [ ]:
!pip install -q "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install -q --no-deps "xformers<0.0.28" "trl>=0.15,<1.0" peft accelerate bitsandbytes
!pip install -q httpx fastapi uvicorn pydantic pandas matplotlib datasets scipy
!pip install -q trackio  # live dashboard; the pipeline still runs without it
!pip install -q --force-reinstall --no-deps "huggingface_hub>=0.34,<1.0"
!pip install openenv-core

# Same install as the L40s/T4 notebooks. Both Colab Pro GPUs are covered:
#   - A100 (Ampere, sm_80, 40 GB, bf16 native)
#   - L4   (Ada Lovelace, sm_89, 22.5 GB, bf16 native)
# The xformers<0.0.28 wheel ships sm_75 + sm_80 + sm_89 in the same release,
# so we don't need a card-specific install path.
#
# We deliberately do NOT install vLLM here (commented out below). It pulls
# `cv2` + `mistral-common` + a long tail of multimodal deps; partial installs
# crash Unsloth's startup hook on import, and on Colab Pro A100 the wall-clock
# saving doesn't justify that risk for a hackathon demo.
# !pip install -q --no-deps vllm  # do NOT enable; see comment above
#
# After this cell finishes, RESTART THE RUNTIME ONCE, then re-run from Cell 1.
# Reason: if `huggingface_hub` was already imported by the runtime before
# these fixes ran, Python has cached the broken module objects and the new
# installs won't take effect until the kernel restarts. (On a fresh Colab
# session this is a no-op.)
print("\nIf this is your FIRST run on a fresh runtime: continue normally.")
print("Otherwise: Runtime -> Restart, then re-run from Cell 1.")

## 2 · Clone the environment repo (Colab)

In [ ]:
!git clone https://github.com/AnirudhChidananda/bioresearch.git
%ls
%cd bioresearch
print("Setup complete")

## 3 · Boot the OpenEnv server

The reward loop talks HTTP to this subprocess. Current OpenEnv releases expose `/health`, `/schema`, `/docs`, and `/openapi.json` — the legacy `/info` is gone.

In [ ]:
import subprocess, time, httpx

server_proc = subprocess.Popen(
    ["uvicorn", "server.app:app", "--host", "127.0.0.1", "--port", "8000"],
    cwd="/content/bioresearch",
    # cwd="/root/bioresearch", #For Modal notebook
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
)

for _ in range(40):
    try:
        if httpx.get("http://127.0.0.1:8000/health", timeout=1.0).status_code == 200:
            print("Server up")
            break
    except Exception:
        time.sleep(1.0)
else:
    raise RuntimeError("OpenEnv server failed to start")

## 4 · Wire up `training_core` + auto-detect GPU + Trackio

This is the only cell that knows we're on Colab Pro. [`training_colab_pro`](../training_colab_pro.py) imports everything from [`training_a100`](../training_a100.py) verbatim and adds two helpers:

- `detect_gpu()` reads `torch.cuda.get_device_name(0)` + capability + free VRAM and returns a `GpuProfile` with a canonical short name (`"A100"`, `"L4"`, `"L40S"`, `"T4"`, ...).
- `auto_train_lab_max_steps(profile)` picks the per-card lab rollout depth (4 on A100/L4, 6 on L40S, 2 on T4). Each lab task does up to this many extra `model.generate` calls per completion, so it's the dominant wall-clock knob after `num_generations`.

Trackio init still degrades gracefully to `report_to="none"` if the install failed — the rest of the notebook produces the same artifacts either way. Set `HF_SPACE_ID` to a `username/space-name` you own to persist the dashboard after the runtime dies.

In [ ]:
import os
import training_core
import training_colab_pro as tcp
from training_core import (
    LEGACY_TASKS, LAB_TASKS, ALL_TASKS, DEFAULT_T4_TASKS,
    TRAIN_LAB_MAX_STEPS, EVAL_LAB_MAX_STEPS,
)

profile = tcp.detect_gpu()
print(f"GPU: {profile.name}  ({profile.raw_name})")
print(f"  VRAM: {profile.vram_gb:.1f} GB  compute_cap: sm_{profile.compute_cap[0]}{profile.compute_cap[1]}  bf16: {profile.bf16_supported}")

# Lab rollout depth scaled to per-card budget (A100/L4=4, L40S=6, T4=2).
TRAIN_LAB_MAX_STEPS = tcp.auto_train_lab_max_steps(profile)
os.environ["TRAIN_LAB_MAX_STEPS"] = str(TRAIN_LAB_MAX_STEPS)
training_core.TRAIN_LAB_MAX_STEPS = TRAIN_LAB_MAX_STEPS
training_core.EVAL_LAB_MAX_STEPS = 12  # quality run -> deeper eval rollouts
print(f"  TRAIN_LAB_MAX_STEPS={TRAIN_LAB_MAX_STEPS}  EVAL_LAB_MAX_STEPS=12")

SERVER_URL = "http://127.0.0.1:8000"
training_core.configure_env(SERVER_URL)

# Optional persistent Trackio dashboard. Leave HF_SPACE_ID unset for a
# local-only dashboard (still visible in Colab via the trackio CLI). The run
# name is templated on the detected card so an A100 run and an L4 run land
# in the same project but as separate runs (easy to compare in the UI).
trackio_cfg = tcp.setup_trackio(
    project="bioresearch-grpo-colab-pro",
    run_name=f"qwen2p5-7b-{profile.name.lower()}-maxepoch",
    space_id=os.environ.get("HF_SPACE_ID"),
    config={
        "model": "Qwen2.5-7B-Instruct-bnb-4bit",
        "lora_r": 32,
        "gpu": profile.name,
        "vram_gb": round(profile.vram_gb, 1),
    },
)
print("Trackio config:", trackio_cfg)

probe = training_core.env_reset(task_type="dna_classification").observation
print(f"Probe OK  task_id={probe.task_id}  question[:60]={probe.question[:60]!r}")

## 5 · Choose the task list

Default trains all 14 tasks — this is the demo run. The 6-task T4 subset and a custom subset are documented as commented lines.

In [ ]:
TASK_LIST = ALL_TASKS                       # 14 reward curves — full demo
# TASK_LIST = DEFAULT_T4_TASKS              # 6-task fast subset for smoke runs
# TASK_LIST = ["dna_reasoning", "clinical_diagnosis", "protein_hypothesis_lab"]

print("Training on:", TASK_LIST)
print("  legacy:", [t for t in TASK_LIST if t in LEGACY_TASKS])
print("  lab   :", [t for t in TASK_LIST if t in LAB_TASKS])

## 6 · Build the mixed-task dataset

`n_per_task=8` (112 rows for all 14 tasks). Reward curves smooth out fine at 8 rows × 14 tasks × 10 generations = ~1120 rollouts per epoch.

In [ ]:
N_PER_TASK = 8
train_ds = training_core.build_mixed_dataset(TASK_LIST, n_per_task=N_PER_TASK, seed=42)
print(train_ds)
print("Sample row keys:", train_ds.column_names)

## 7 · Load Qwen2.5-7B + LoRA (r=32)

Hard-pinned to **Qwen2.5-7B-Instruct-bnb-4bit**. See the rationale in Cell 0 — we don't carry a 14B path because that would double cost for marginal lift while leaving no VRAM headroom.

After loading we print VRAM headroom and assert >30 GB free — if the assertion trips, something has changed in the deps stack and we want to know loudly before training.

In [ ]:
from unsloth import FastLanguageModel
import torch

MODEL_NAME = "unsloth/Qwen2.5-7B-Instruct-bnb-4bit"          # fixed for Colab Pro quality run
# MODEL_NAME = "unsloth/Qwen2.5-1.5B-Instruct-bnb-4bit"      # smoke test only — DO NOT use for the demo run
MAX_SEQ_LEN = 3072

# Pin the compute dtype so the LoRA matmul kernel doesn't trip the
# "self and mat2 must have the same dtype" RuntimeError. bf16 on
# A100/L4/L40S/H100 (all sm_80+), fp16 on T4/V100. The GRPOConfig built in
# Cell 20 from `tcp.auto_grpo_kwargs` uses the same rule, so bf16/fp16 stays
# consistent end-to-end.
TARGET_DTYPE = torch.bfloat16 if profile.bf16_supported else torch.float16

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=MAX_SEQ_LEN,
    load_in_4bit=True,
    dtype=TARGET_DTYPE,
)
model = FastLanguageModel.get_peft_model(
    model,
    r=32,
    lora_alpha=64,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    use_gradient_checkpointing="unsloth",
    random_state=42,
)

# Belt-and-suspenders: force every trainable LoRA parameter to TARGET_DTYPE.
# Recent Unsloth + TRL GRPO + 4-bit base sometimes leaves the LoRA A/B matrices
# in fp32 even when the base model is fp16/bf16 — the first GRPO backward then
# crashes inside `unsloth/kernels/utils.py::matmul_lora` with:
#   RuntimeError: self and mat2 must have the same dtype, but got Half and Float
# Casting them here keeps the kernel happy without touching the (already-correct)
# 4-bit base weights.
_n_cast = 0
for _name, _p in model.named_parameters():
    if _p.requires_grad and _p.dtype == torch.float32:
        _p.data = _p.data.to(TARGET_DTYPE)
        _n_cast += 1
if _n_cast:
    print(f"Cast {_n_cast} fp32 LoRA params -> {TARGET_DTYPE}")

tokenizer.padding_side = "left"
training_core.configure_model(model, tokenizer, max_new_tokens=320)

# Same fix as train_grpo_t4.ipynb Cell 14: kill Qwen2.5's baked-in
# `generation_config.max_length=32768` so HuggingFace's "both max_new_tokens and
# max_length seem to have been set" warning never fires. Without this it would
# spam ~one warning per generate call (~thousands over a 240-step run).
if model.generation_config is not None:
    model.generation_config.max_length = None

if torch.cuda.is_available():
    free, total = torch.cuda.mem_get_info()
    print(f"VRAM headroom after model load: {free / 1e9:.1f} GB free / {total / 1e9:.1f} GB total")
    # Card-aware floor: L4 has 22.5 GB total and 7B+r32 LoRA leaves ~15-17 GB
    # free; A100/L40S/H100 should leave >25 GB. We assert against whichever
    # card we actually got (default to the strict A100/L40S/H100 floor).
    min_free_gb = 14 if profile.name == "L4" else 25
    assert free > min_free_gb * 1e9, (
        f"Only {free / 1e9:.1f} GB free on {profile.name} after loading 7B + LoRA r=32 — "
        f"expected >{min_free_gb} GB. Something in the deps stack regressed; check "
        "4-bit loading + LoRA r=32."
    )
print("Model + tokenizer loaded and registered with training_core")

## 8 · Baseline rollouts (BEFORE training)

Run the *untrained* policy on a fixed seed of 5 held-out `task_id`s per task. We capture the pool here and reuse it after training so the before/after compare on identical briefs.

In [ ]:
FastLanguageModel.for_inference(model)        # 2x faster greedy-ish gen

baseline_rows, eval_pool = tcp.collect_rollouts_with_pool(
    TASK_LIST, n_per_task=5, seed=2026,
)
tcp.save_rollouts(baseline_rows, "baseline_rollouts.json")

# Per-task summary so we have a number to point at on the slide.
import statistics
print(f"Baseline rollouts (n={len(baseline_rows)}):")
for task in TASK_LIST:
    rs = [r["reward"] for r in baseline_rows if r["task_type"] == task]
    if rs:
        print(f"  {task:30s}  mean={statistics.fmean(rs):.3f}  n={len(rs)}")

FastLanguageModel.for_training(model)         # back to training mode

## 9 · Reward-distribution diagnostic

Score a hand-curated bag of completions per task through the reward function. The point isn't to evaluate the model — the completions are canned. The point is to **prove to the judges** that the reward signal:

1. Spans a meaningful range (not all 0 or 1).
2. Reflects completion quality (correct answers get high reward, garbage gets low).
3. Has variance for GRPO's relative advantage to actually optimize against.

In [ ]:
import matplotlib.pyplot as plt

diag_df = tcp.reward_distribution_diagnostic(TASK_LIST, n_samples_per_task=4)
print(diag_df)

fig, ax = plt.subplots(figsize=(12, 4))
tasks_in_order = [t for t in TASK_LIST if t in set(diag_df["task_type"])]
data = [diag_df[diag_df["task_type"] == t]["reward"].tolist() for t in tasks_in_order]
ax.boxplot(data, labels=tasks_in_order, vert=True, showmeans=True)
ax.set_ylabel("Reward (canned-completion bag)")
ax.set_title("Reward signal has variance — GRPO has something to optimize")
ax.set_ylim(0, 1)
plt.xticks(rotation=45, ha="right")
plt.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.savefig("reward_diagnostic.png", dpi=140)
plt.show()

## 10 · `GRPOConfig` + `GRPOTrainer` (auto-tuned, Trackio-wired)

The whole config dict comes out of [`tcp.auto_grpo_kwargs(profile, trackio_cfg)`](../training_colab_pro.py). It tunes the run on **four independent dimensions** based on the GPU detected in Cell 8:

| Dimension | A100 preset | L4 preset | Why per-card |
|-----------|-------------|-----------|--------------|
| **`num_generations`** | 12 | 8 | GRPO's advantage variance scales `1/N`. A100's 40 GB hosts the KV cache for N=12; L4's 22.5 GB taps out at N=8. |
| **`max_completion_length`** | 768 | 640 | Qwen 7B's JSON action+reasoning fits under ~700 tokens for this env, so 768 is the comfort ceiling and 640 is the L4 budget. |
| **`max_steps`** | 240 | 180 | Paired with the dataset (112 prompts at `n_per_task=8`) and 4 prompts/step → **~8.6 epochs on A100, ~6.4 on L4**. Both inside the no-overfit regime for GRPO with KL `beta=0.04`. |
| **Cosine LR + 5% warmup** | yes | yes | Smoother than constant LR on long runs (>200 steps), where late-stage updates can otherwise destabilise the policy. The L40s/A100/T4 notebooks use constant LR — for *this* run length cosine empirically gives flatter KL and tighter reward variance. |

**`save_steps=30`** lands a checkpoint every ~12 min on A100 / ~17 min on L4. Colab Pro sessions occasionally drop without warning, so this is the resilience knob — at most one checkpoint of work is lost per disconnect, and Cell 21 below explains how to resume from the latest `checkpoint-XXX/` directory.

**`USE_VLLM`** auto-detects vLLM and enables colocated rollouts. On Colab Pro we deliberately skip the vLLM install in Cell 1 (cv2 / mistral-common dependency hell), so this will normally print "vLLM not available" and use native HF generation — still fast enough at the wall-clock targets in Cell 0.

**`LabShapingCallback`** drains the per-step lab rollout log on every `on_log` event so the dashboard shows process vs terminal reward decomposition for each lab task.

In [ ]:
import torch
from trl import GRPOConfig, GRPOTrainer

reward_fns = [training_core.make_reward_fn(t) for t in TASK_LIST]
print("Reward functions wired:")
for fn in reward_fns:
    print(" -", fn.__name__)

# Optional vLLM rollouts — ~2-3x faster generation when wheels are present.
# We deliberately don't install vLLM in Cell 1 (cv2 + mistral-common dep hell),
# so this normally falls through to native HF generation.
USE_VLLM = False
try:
    import vllm  # noqa: F401
    USE_VLLM = True
    print("vLLM detected -> enabling fast colocated rollouts")
except Exception:
    print("vLLM not available -> using native HF generation (expected on Colab Pro)")

# Auto-tune everything from the detected GPU profile from Cell 8.
# Returns max_steps / num_generations / max_completion_length / max_prompt_length
# / per_device_train_batch_size / gradient_accumulation_steps / cosine LR /
# warmup_ratio=0.05 / save_steps=30 / bf16-by-cap, plus splats `trackio_cfg` in.
_grpo_kwargs = tcp.auto_grpo_kwargs(
    profile,
    trackio_cfg,
    output_dir="grpo_bioresearch_colab_pro",
)
_grpo_kwargs["use_vllm"] = USE_VLLM
if USE_VLLM:
    _grpo_kwargs["vllm_mode"] = "colocate"  # single-GPU Colab

# Forward-compat: older TRL versions don't know `vllm_mode`, newer don't know `use_vllm`.
# Strip unknowns until GRPOConfig accepts the dict.
while True:
    try:
        grpo_config = GRPOConfig(**_grpo_kwargs)
        break
    except TypeError as exc:
        msg = str(exc)
        dropped = False
        for k in ("vllm_mode", "use_vllm"):
            if k in _grpo_kwargs and k in msg:
                print(f"[GRPOConfig] dropping unsupported kwarg {k!r} ({msg.splitlines()[0]})")
                _grpo_kwargs.pop(k)
                dropped = True
                break
        if not dropped:
            raise

# Receipts so the run is reproducible from the notebook output alone.
_dataset_size = len(train_ds)
_prompts_per_step = grpo_config.per_device_train_batch_size * grpo_config.gradient_accumulation_steps
_epochs = (grpo_config.max_steps * _prompts_per_step) / max(_dataset_size, 1)
print(f"\nResolved GRPOConfig for {profile.name}:")
print(f"  max_steps              = {grpo_config.max_steps}")
print(f"  num_generations        = {grpo_config.num_generations}")
print(f"  max_completion_length  = {grpo_config.max_completion_length}")
print(f"  max_prompt_length      = {grpo_config.max_prompt_length}")
print(f"  prompts/step           = {_prompts_per_step}  (pdtbs={grpo_config.per_device_train_batch_size} x ga={grpo_config.gradient_accumulation_steps})")
print(f"  dataset size           = {_dataset_size}")
print(f"  -> epochs              = {_epochs:.2f}")
print(f"  lr_scheduler_type      = {grpo_config.lr_scheduler_type}")
print(f"  warmup_ratio           = {grpo_config.warmup_ratio}")
print(f"  save_steps             = {grpo_config.save_steps}")
print(f"  bf16 / fp16            = {grpo_config.bf16} / {grpo_config.fp16}")

trainer = GRPOTrainer(
    model=model,
    processing_class=tokenizer,
    args=grpo_config,
    train_dataset=train_ds,
    reward_funcs=reward_fns,
    callbacks=[tcp.make_lab_shaping_callback()],
)

# ------------------------------------------------------------------
# Post-trainer-init dtype reconciliation.
#
# Two things conspire to cause:
#     RuntimeError: self and mat2 must have the same dtype, but got Half and Float
# inside `unsloth/kernels/utils.py::matmul_lora` on the first GRPO backward:
#
# 1. PEFT defaults to `autocast_adapter_dtype=True`, which promotes the LoRA
#    A/B matrices to fp32 every time the model is (re-)wrapped — including
#    inside `GRPOTrainer.__init__`. So the Cell 14 cast gets undone here.
# 2. Unsloth's matmul_lora kernel does NOT promote inputs; it asserts equal
#    dtypes and crashes if base activations (Half/BFloat16) and LoRA weights
#    (Float) disagree.
#
# The fix is to cast trainable params back to the base-model dtype AFTER the
# trainer has wrapped the model. This is idempotent and adds <50ms.
# ------------------------------------------------------------------

# Find the actual base activation dtype by looking at a non-trainable weight.
# Most reliable: take the first 4-bit dequantize-target weight that exists.
_base_dtype = None
for _n, _p in trainer.model.named_parameters():
    if not _p.requires_grad and _p.dtype in (torch.float16, torch.bfloat16):
        _base_dtype = _p.dtype
        break
if _base_dtype is None:
    # Fallback: trust the GRPOConfig flags.
    _base_dtype = torch.bfloat16 if grpo_config.bf16 else torch.float16
print(f"Base model compute dtype detected as: {_base_dtype}")

# Cast every trainable (LoRA) param to the base dtype.
_n_cast = 0
_n_kept = 0
for _n, _p in trainer.model.named_parameters():
    if not _p.requires_grad:
        continue
    if _p.dtype != _base_dtype:
        _p.data = _p.data.to(_base_dtype)
        _n_cast += 1
    else:
        _n_kept += 1
print(f"Reconciled LoRA dtypes: cast {_n_cast} -> {_base_dtype}, "
      f"kept {_n_kept} already-matching params")

# Also align the GRPOConfig precision flags with the actual base dtype, so
# autocast contexts inside GRPOTrainer don't re-introduce the mismatch.
trainer.args.bf16 = (_base_dtype == torch.bfloat16)
trainer.args.fp16 = (_base_dtype == torch.float16)
print(f"Aligned trainer.args: bf16={trainer.args.bf16}  fp16={trainer.args.fp16}")

print("Trainer ready")

## 11 · Train

Expected wall-clock for the full 14-task run on Colab Pro:

| GPU | Wall-clock target | Steps × per-step time |
|-----|-------------------|------------------------|
| **A100 40 GB** | **~3 hours** | 240 × ~45-55 s |
| **L4 22.5 GB**  | **~2 h 30 m** | 180 × ~50-55 s |

Both fit inside Colab Pro's session window with eval (Cell 24) headroom. Watch the live Trackio dashboard for early warning signs — if reward variance collapses to zero across all tasks before step 50, abort and check the reward diagnostic.

### If Colab disconnects mid-run

Checkpoints land every 30 steps in `grpo_bioresearch_colab_pro/checkpoint-XXX/`. To resume after a disconnect:

1. Re-run Cells 1-20 on the new runtime (this rebuilds the trainer with the same config).
2. Replace `trainer.train()` below with:

   ```python
   import glob, os
   ckpts = sorted(glob.glob("grpo_bioresearch_colab_pro/checkpoint-*"),
                  key=lambda p: int(p.rsplit("-", 1)[-1]))
   trainer.train(resume_from_checkpoint=ckpts[-1] if ckpts else None)
   ```

3. The lost work is at most one checkpoint (~12 min on A100, ~17 min on L4).

In [ ]:
trainer.train()
tcp.trackio_finish()  # noop if Trackio is not in use

## 12 · Trained rollouts (AFTER training)

Same `task_ids` as Cell 8 — `eval_pool` pins them. This is what makes the before/after table a *paired* comparison instead of two unpaired draws.

In [ ]:
FastLanguageModel.for_inference(model)

# Quality run: n_per_task=5 matches the baseline pool exactly (paired comparison
# stays paired). The L40s notebook also uses 5 here; the T4 notebook drops to 3
# only because it's a budget run.
trained_rows = tcp.collect_rollouts(
    TASK_LIST, n_per_task=5, seed=2026, task_id_pool=eval_pool,
)
tcp.save_rollouts(trained_rows, "trained_rollouts.json")

import statistics
print(f"Trained rollouts (n={len(trained_rows)}):")
for task in TASK_LIST:
    rs = [r["reward"] for r in trained_rows if r["task_type"] == task]
    if rs:
        print(f"  {task:30s}  mean={statistics.fmean(rs):.3f}  n={len(rs)}")

## 13 · Before-vs-after table + bar chart

Headline judging artifact #1. Per-task delta in mean reward, sorted by biggest win first. The bar chart makes the wins obvious at a glance.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

table = tcp.before_after_table(baseline_rows, trained_rows)
print(table.to_string(index=False) if hasattr(table, "to_string") else table)

if hasattr(table, "iloc"):
    tasks = table["task_type"].tolist()
    deltas = table["delta"].tolist()
    baselines = table["baseline_mean"].tolist()
    traineds = table["trained_mean"].tolist()
else:
    tasks = [r["task_type"] for r in table]
    deltas = [r["delta"] for r in table]
    baselines = [r["baseline_mean"] for r in table]
    traineds = [r["trained_mean"] for r in table]

x = np.arange(len(tasks))
fig, ax = plt.subplots(figsize=(13, 5))
width = 0.38
ax.bar(x - width/2, baselines, width, label="Baseline (untrained)")
ax.bar(x + width/2, traineds, width, label="After GRPO training")
for i, d in enumerate(deltas):
    color = "green" if d > 0 else ("red" if d < 0 else "gray")
    ax.annotate(f"{d:+.2f}", xy=(x[i], max(baselines[i], traineds[i]) + 0.02),
                ha="center", color=color, fontweight="bold")
ax.set_xticks(x); ax.set_xticklabels(tasks, rotation=40, ha="right")
ax.set_ylabel("Mean reward")
ax.set_title(f"Before vs After GRPO — per-task mean reward (paired task_ids, Colab Pro {profile.name} + Qwen2.5-7B)")
ax.set_ylim(0, 1)
ax.legend()
ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.savefig("before_after_bar.png", dpi=140)
plt.show()

## 14 · Sample rollout transcripts (the headline demo artifact)

Headline judging artifact #2. For each task, the 2 `task_id`s with the largest positive delta are shown side-by-side: prompt → baseline completion + reward → trained completion + reward. Judges can literally read the qualitative jump.

In [ ]:
from IPython.display import Markdown, display

transcripts_md = tcp.render_sample_transcripts(
    baseline_rows, trained_rows, k=2, max_chars_per_block=600,
)
with open("sample_transcripts.md", "w") as f:
    f.write(transcripts_md)
print(f"Wrote sample_transcripts.md ({len(transcripts_md)} chars)")

display(Markdown(transcripts_md))

## 15 · Per-task reward curves (from `trainer.state.log_history`)

Live Trackio is the primary instrument; this is the static fallback so the artifact survives the runtime ending. Each curve is a single task's reward function — `0.0` rows (rows from other tasks) are masked before smoothing.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

log_df = pd.DataFrame(trainer.state.log_history)
print("Reward-related columns:", [c for c in log_df.columns if "reward" in c])

fig, ax = plt.subplots(figsize=(12, 5))
for task in TASK_LIST:
    candidates = [c for c in log_df.columns
                  if task in c and "reward" in c and "std" not in c.lower() and "process" not in c and "terminal" not in c]
    if not candidates:
        continue
    col = candidates[0]
    subset = log_df[["step", col]].dropna()
    subset = subset[subset[col] > 0.0].reset_index(drop=True)
    if subset.empty:
        continue
    smooth = subset[col].rolling(window=10, min_periods=1).mean()
    ax.plot(subset["step"], smooth, linewidth=2.0, label=task)

ax.set_xlabel("GRPO step")
ax.set_ylabel("Reward (10-step EMA, gated rows only)")
ax.set_title(f"Per-task reward curves — Colab Pro {profile.name} (Qwen2.5-7B, num_generations={grpo_config.num_generations}, max_steps={grpo_config.max_steps})")
ax.legend(loc="best", fontsize=8, ncol=2)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig("training_curves.png", dpi=140)
plt.show()

## 16 · Save LoRA + (optional) Hub push + teardown

The LoRA adapter is **~150 MB** on disk (Qwen 7B base + r=32 over 7 target modules). On a normal Colab Pro connection the `push_to_hub` call below typically completes in **~30 seconds**. Set `HF_PUSH_REPO=username/repo-name` and `HF_TOKEN=hf_...` before running this cell to enable the upload.

In [ ]:
model.save_pretrained("grpo_bioresearch_colab_pro_lora")
tokenizer.save_pretrained("grpo_bioresearch_colab_pro_lora")
print("LoRA saved to grpo_bioresearch_colab_pro_lora/  (~150 MB)")

# Optional: push to the Hub if HF_TOKEN is set in the env.
HF_REPO = os.environ.get("HF_PUSH_REPO")
if HF_REPO and os.environ.get("HF_TOKEN"):
    try:
        model.push_to_hub(HF_REPO, token=os.environ["HF_TOKEN"])
        tokenizer.push_to_hub(HF_REPO, token=os.environ["HF_TOKEN"])
        print(f"Pushed to https://huggingface.co/{HF_REPO}")
    except Exception as e:
        print(f"Hub push failed ({type(e).__name__}: {e}); LoRA still saved locally.")
else:
    print("Skipped Hub push (set HF_PUSH_REPO + HF_TOKEN to enable).")

server_proc.terminate()
try:
    server_proc.wait(timeout=5)
except Exception:
    server_proc.kill()
print("Server terminated")